# Exercise 9 — Beer Delivery


## Task

A Berlin trading company delivers beer using a truck with five compartments. Each compartment has a fixed capacity and can carry **at most one beer type**.

### Compartment capacities

| Compartment | Capacity (liters) |
|---|---:|
| 1 | 2700 |
| 2 | 2800 |
| 3 | 1100 |
| 4 | 1800 |
| 5 | 3400 |

The company must serve an order for three beer brands. Unmet demand incurs a per-liter penalty, but shortage is limited for each brand. Quantities are counted in full liters.

| Beer | Demand (liters) | Penalty per liter | Maximum shortage |
|---|---:|---:|---:|
| Berliner Pilsner | 2900 | 10 | 500 |
| Berliner Kindl | 4000 | 8 | 500 |
| Schultheiss | 4900 | 6 | 300 |

Tasks:

1. Formulate the problem algebraically as a mixed-integer model.
2. Translate it into JuMP and solve it.
3. Interpret the compartment assignment and any remaining shortage.


## Reference implementation


In [1]:
using HiGHS, JuMP

compartments = collect(1:5) # compartments
brands = ["pilsener", "kindel", "schultheiss"] # brands

s = Dict(zip(compartments, [2700, 2800, 1100, 1800, 3400]))
d = Dict(zip(brands, [2900, 4000, 4900]))
ms = Dict(zip(brands, [500, 500, 300]))
p = Dict(zip(brands, [10, 8, 6]))

m = Model(HiGHS.Optimizer)
@variable(m, x[compartments, brands], Bin)
@variable(m, q[brands] >= 0, Int) # penalty variable for brands
@constraint(
    m,
    market_clearing[b=brands],
    sum(x[c, b] * s[c] for c in compartments) >= d[b] - ms[b]
)
@constraint(m, penalty[b=brands], q[b] >= d[b] - sum(x[c, b]*s[c] for c in compartments))
@constraint(m, compartments[c=compartments], sum(x[c, b] for b in brands) <= 1)
@objective(m, Min, sum(p[b]*q[b] for b in brands))
optimize!(m)
objective_value(m)

value.(x)
value.(q)


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
MIP has 11 rows; 18 cols; 48 nonzeros; 18 integer variables (15 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+03]
  Cost    [6e+00, 1e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 5e+03]
Presolving model
11 rows, 18 cols, 48 nonzeros 0s
9 rows, 10 cols, 25 nonzeros 0s
5 rows, 7 cols, 18 nonzeros 0s
5 rows, 7 cols, 18 nonzeros 0s
Presolve reductions: rows 5(-6); columns 7(-11); nonzeros 18(-30) 
Objective function is integral with scale 0.5

Solving MIP model with:
   5 rows
   7 cols (5 binary, 2 integer, 0 implied int., 0 continuous, 0 domain fixed)
   18 nonzeros
   Thread count 5 (of 10 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P 

1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, ["pilsener", "kindel", "schultheiss"]
And data, a 3-element Vector{Float64}:
 100.0
 200.0
   0.0